# Cost-Sensitive Profit Optimization

The headline number — built from the cost matrix, not accuracy/AUC.

**Cost matrix (per dollar of principal):**

| Predicted ↓ / Actual → | Good | Default |
|---|---|---|
| Approve | `+r·t − OpEx` (TN value) | `−LGD − OpEx` (FN cost) |
| Reject  | `−(r·t − OpEx)` (FP opp. cost) | 0 (TP) |

**Optimal threshold** for a given loan is derived from the cost matrix:

$$\text{PD}^* = \frac{r \cdot t - \text{OpEx}}{\text{LGD} + r \cdot t}$$

Approve iff predicted PD < PD*. This is what `compare_to_naive` enforces. Naive baseline = approve every applicant.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np

from src import data_loader, metrics as M, model as Mdl

trained = Mdl.load_model()
df = data_loader.load_clean()
df['pd'] = trained.predict_pd(df)

stats = M.compare_to_naive(df)
print(M.headline_summary(stats))
stats

## Cost matrix for a sample loan

In [ ]:
cm = M.per_loan_cost_matrix(loan_amnt=15000, int_rate_pct=13, term_months=36)
print('TN value (approve a good loan)  : $', round(cm.tn_value, 2))
print('FP cost  (reject a good loan)   : $', round(cm.fp_cost, 2))
print('FN cost  (approve a defaulter)  : $', round(cm.fn_cost, 2))
print()
print('Analytical optimal threshold    : ',
      round(M.optimal_threshold_analytical(13, 36), 4))

## Profit curve — sweep a single global threshold

In [ ]:
sweep = M.sweep_threshold(df)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sweep['threshold'], sweep['profit'] / 1e6, color='#2c3e50', lw=2)
ax.set_xlabel('Global PD threshold (approve if PD < t)')
ax.set_ylabel('Total portfolio profit ($M)')
ax.set_title('Profit curve across global thresholds', fontweight='bold')
best = M.find_optimal_fixed_threshold(df)
ax.axvline(best['threshold'], color='#e74c3c', linestyle='--',
           label=f"Best fixed t = {best['threshold']:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/profit_curve.png', dpi=140, bbox_inches='tight')
plt.show()
best